In [ ]:
!pip install numpy pandas scikit-learn tensorflow matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report, matthews_corrcoef
from sklearn.model_selection import LeaveOneGroupOut

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, Conv1D,
    GlobalAveragePooling1D, Concatenate, Multiply, Reshape
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
FEATURE_NAMES = [
    "Jitter_local", "Jitter_local_abs", "Jitter_rap", "Jitter_ppq5", "Jitter_ddp",
    "Shimmer_local", "Shimmer_local_dB", "Shimmer_apq3", "Shimmer_apq5",
    "Shimmer_apq11", "Shimmer_dda",
    "AC", "NTH", "HTN",
    "Median_pitch", "Mean_pitch", "Std_pitch", "Min_pitch", "Max_pitch",
    "Num_pulses", "Num_periods", "Mean_period", "Std_period",
    "Frac_unvoiced_frames", "Num_voice_breaks", "Degree_voice_breaks"
]

print("train_data.txt dosyasını yükle:")
uploaded = files.upload()
TRAIN_PATH = list(uploaded.keys())[0]

df = pd.read_csv(TRAIN_PATH, header=None)
df.columns = ["subject_id"] + FEATURE_NAMES + ["UPDRS", "class"]

print(df.head())
print("Veri boyutu:", df.shape)
print("Eksik değer sayısı:", df.isnull().sum().sum())
print("Sınıf dağılımı:")
print(df["class"].value_counts())
print("Subject sayısı:", df["subject_id"].nunique())

train_data.txt dosyasını yükle:


Saving train_data.txt to train_data.txt
   subject_id  Jitter_local  Jitter_local_abs  Jitter_rap  Jitter_ppq5  \
0           1         1.488          0.000090       0.900        0.794   
1           1         0.728          0.000038       0.353        0.376   
2           1         1.220          0.000074       0.732        0.670   
3           1         2.502          0.000123       1.156        1.634   
4           1         3.509          0.000167       1.715        1.539   

   Jitter_ddp  Shimmer_local  Shimmer_local_dB  Shimmer_apq3  Shimmer_apq5  \
0       2.699          8.334             0.779         4.517         4.609   
1       1.059          5.864             0.642         2.058         3.180   
2       2.196          8.719             0.875         4.347         5.166   
3       3.469         13.513             1.273         5.263         8.771   
4       5.145          9.112             1.040         3.102         4.927   

   ...  Max_pitch  Num_pulses  Num_periods  Me

In [ ]:
def add_sample_info(df):
    df = df.copy()
    df["sample_no"] = df.groupby("subject_id").cumcount() + 1

    def sample_type(s):
        if s in [1, 2, 3]:
            return "vowel"
        elif 4 <= s <= 13:
            return "number"
        else:
            return "other"

    df["sample_type"] = df["sample_no"].apply(sample_type)
    return df


df = add_sample_info(df)

df[["subject_id", "sample_no", "sample_type", "class"]].head(30)

,subject_id,sample_no,sample_type,class
0,1,1,vowel,1
1,1,2,vowel,1
2,1,3,vowel,1
3,1,4,number,1
4,1,5,number,1
5,1,6,number,1
6,1,7,number,1
7,1,8,number,1
8,1,9,number,1
9,1,10,number,1


In [ ]:
def weighted_subject_summary(
    df,
    vowel_w=1.5,
    number_w=1.2,
    other_w=1.0,
    eps=1e-8
):
    rows = []

    type_weights = {
        "vowel": vowel_w,
        "number": number_w,
        "other": other_w
    }

    for subject_id, g in df.groupby("subject_id"):
        X = g[FEATURE_NAMES].values.astype(float)

        median = np.median(X, axis=0)
        mad = np.median(np.abs(X - median), axis=0) + eps

        robust_z = np.abs((X - median) / mad)
        anomaly_score = np.mean(robust_z, axis=1)

        robust_weights = np.exp(-anomaly_score)
        sample_weights = g["sample_type"].map(type_weights).values

        weights = robust_weights * sample_weights
        weights = weights / (weights.sum() + eps)

        wmean = np.average(X, axis=0, weights=weights)

        variance = np.average((X - wmean) ** 2, axis=0, weights=weights)
        wstd = np.sqrt(variance)

        row = {
            "subject_id": subject_id,
            "class": int(g["class"].iloc[0])
        }

        for i, name in enumerate(FEATURE_NAMES):
            row[name + "_wmean"] = wmean[i]
            row[name + "_wstd"] = wstd[i]

        rows.append(row)

    return pd.DataFrame(rows)


summary_df = weighted_subject_summary(
    df,
    vowel_w=1.5,
    number_w=1.2,
    other_w=1.0
)

summary_df.head()

,subject_id,class,Jitter_local_wmean,Jitter_local_wstd,Jitter_local_abs_wmean,Jitter_local_abs_wstd,Jitter_rap_wmean,Jitter_rap_wstd,Jitter_ppq5_wmean,Jitter_ppq5_wstd,...,Mean_period_wmean,Mean_period_wstd,Std_period_wmean,Std_period_wstd,Frac_unvoiced_frames_wmean,Frac_unvoiced_frames_wstd,Num_voice_breaks_wmean,Num_voice_breaks_wstd,Degree_voice_breaks_wmean,Degree_voice_breaks_wstd
0,1,1,1.965055,0.498546,0.000098,0.000023,0.958372,0.299346,1.070075,0.373700,...,0.005014,0.000340,0.000369,0.000136,11.438926,10.301648,0.038206,0.197893,0.173001,0.983682
1,2,1,2.662243,0.739437,0.000219,0.000061,1.237477,0.393310,1.394454,0.563906,...,0.008199,0.000275,0.000616,0.000194,7.125314,7.887225,0.000000,0.000000,0.000000,0.000000
2,3,1,2.278831,0.753062,0.000107,0.000032,1.243291,0.413341,1.258544,0.399174,...,0.004746,0.000288,0.000498,0.000288,31.979004,20.550509,0.954779,1.309728,13.294886,15.455983
3,4,1,1.380933,0.504486,0.000057,0.000020,0.722464,0.271751,0.733354,0.302153,...,0.004182,0.000258,0.000248,0.000105,8.632154,10.091530,0.565288,0.633868,8.195056,10.127015
4,5,1,2.593978,0.951326,0.000235,0.000093,1.147006,0.516743,1.235853,0.503373,...,0.009017,0.000693,0.000716,0.000272,23.337799,21.093733,0.609887,0.789376,6.831017,10.048792


In [ ]:
SUMMARY_FEATURES = [c for c in summary_df.columns if c not in ["subject_id", "class"]]

X = summary_df[SUMMARY_FEATURES].values.astype(np.float32)
y = summary_df["class"].values.astype(int)
groups = summary_df["subject_id"].values

print("Subject-level veri:", X.shape)
print("Sınıf dağılımı:", np.bincount(y))

Subject-level veri: (40, 52)
Sınıf dağılımı: [20 20]


In [ ]:
def se_block(x, ratio=4):
    channels = x.shape[-1]

    se = Dense(max(channels // ratio, 1), activation="relu")(x)
    se = Dense(channels, activation="sigmoid")(se)

    return Multiply()([x, se])


def build_grouped_cnn(input_dim, lr=1e-3, dropout=0.4, l2_val=1e-4):
    inp = Input(shape=(input_dim,), name="input_features")

    x_reshaped = Reshape((input_dim, 1))(inp)

    conv = Conv1D(
        filters=16,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(x_reshaped)

    conv = BatchNormalization()(conv)

    conv = Conv1D(
        filters=16,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(conv)

    conv = BatchNormalization()(conv)

    pooled = GlobalAveragePooling1D()(conv)

    att = se_block(pooled, ratio=4)

    fusion = Concatenate()([att, inp])

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(fusion)

    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)

    x = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(x)

    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)

    out = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inp, outputs=out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


model_test = build_grouped_cnn(input_dim=X.shape[1])
model_test.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_features      │ (None, 52)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 52, 1)     │          0 │ input_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 52, 16)    │         64 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 52, 16)    │         64 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 52, 16)    │        784 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 16)    │         64 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │         68 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         80 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 16)        │          0 │ global_average_p… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 68)        │          0 │ multiply[0][0],   │
│ (Concatenate)       │                   │            │ input_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,416 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,037 (31.39 KB)

 Trainable params: 7,781 (30.39 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
def train_loso_cnn(X, y, groups, k_features=15):
    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    y_prob_all = []

    selected_features_all = []

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        print("\n" + "="*60)
        print(f"FOLD {fold}/{len(np.unique(groups))}")
        print("="*60)

        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)

        selector = SelectKBest(
            score_func=mutual_info_classif,
            k=min(k_features, X_train_scaled.shape[1])
        )

        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)

        selected_idx = selector.get_support(indices=True)
        selected_names = [SUMMARY_FEATURES[i] for i in selected_idx]
        selected_features_all.extend(selected_names)

        model = build_grouped_cnn(
            input_dim=X_train_sel.shape[1],
            lr=1e-3,
            dropout=0.45,
            l2_val=1e-4
        )

        callbacks = [
            EarlyStopping(
                monitor="loss",
                patience=20,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="loss",
                patience=8,
                factor=0.5,
                min_lr=1e-5
            )
        ]

        model.fit(
            X_train_sel,
            y_train,
            epochs=200,
            batch_size=8,
            verbose=0,
            callbacks=callbacks
        )

        y_prob = model.predict(X_test_sel, verbose=0).ravel()
        y_pred = (y_prob >= 0.5).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
        y_prob_all.extend(y_prob)

        acc = accuracy_score(y_test, y_pred)

        print("Test subject:", groups[test_idx][0])
        print("Gerçek:", y_test[0], "Tahmin:", y_pred[0], "Olasılık:", y_prob[0])
        print("Fold accuracy:", acc)

        fold_results.append({
            "fold": fold,
            "subject_id": groups[test_idx][0],
            "true": y_test[0],
            "pred": y_pred[0],
            "prob": y_prob[0],
            "acc": acc
        })

    return (
        np.array(y_true_all),
        np.array(y_pred_all),
        np.array(y_prob_all),
        pd.DataFrame(fold_results),
        selected_features_all
    )

In [ ]:

def build_grouped_cnn(input_dim, lr=1e-3, dropout=0.45, l2_val=1e-4):
    inp = Input(shape=(input_dim,), name="input_features")

    x_reshaped = Reshape((input_dim, 1))(inp)

    conv = Conv1D(
        filters=8,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(x_reshaped)

    conv = BatchNormalization()(conv)

    conv = Conv1D(
        filters=8,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(conv)

    conv = BatchNormalization()(conv)

    pooled = GlobalAveragePooling1D()(conv)

    att = se_block(pooled, ratio=4)

    fusion = Concatenate()([att, inp])

    x = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(fusion)

    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)

    x = Dense(
        16,
        activation="relu",
        kernel_regularizer=l2(l2_val)
    )(x)

    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)

    out = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inp, outputs=out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def train_loso_cnn(
    X,
    y,
    groups,
    k_features=10,
    threshold=0.40,
    pd_weight=1.2
):
    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    y_prob_all = []

    selected_features_all = []
    fold_results = []

    total_folds = len(np.unique(groups))

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        print("\n" + "="*60)
        print(f"FOLD {fold}/{total_folds}")
        print("="*60)

        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)

        selector = SelectKBest(
            score_func=mutual_info_classif,
            k=min(k_features, X_train_scaled.shape[1])
        )

        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)

        selected_idx = selector.get_support(indices=True)
        selected_names = [SUMMARY_FEATURES[i] for i in selected_idx]
        selected_features_all.extend(selected_names)

        model = build_grouped_cnn(
            input_dim=X_train_sel.shape[1],
            lr=1e-3,
            dropout=0.45,
            l2_val=1e-4
        )

        callbacks = [
            EarlyStopping(
                monitor="loss",
                patience=20,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="loss",
                patience=8,
                factor=0.5,
                min_lr=1e-5
            )
        ]

        model.fit(
            X_train_sel,
            y_train,
            epochs=200,
            batch_size=8,
            verbose=0,
            callbacks=callbacks,
            class_weight={0: 1.0, 1: pd_weight}
        )

        y_prob = model.predict(X_test_sel, verbose=0).ravel()
        y_pred = (y_prob >= threshold).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
        y_prob_all.extend(y_prob)

        acc = accuracy_score(y_test, y_pred)

        print("Test subject:", groups[test_idx][0])
        print("Gerçek:", y_test[0], "Tahmin:", y_pred[0], "Olasılık:", y_prob[0])
        print("Fold accuracy:", acc)

        fold_results.append({
            "fold": fold,
            "subject_id": groups[test_idx][0],
            "true": y_test[0],
            "pred": y_pred[0],
            "prob": y_prob[0],
            "acc": acc
        })

    return (
        np.array(y_true_all),
        np.array(y_pred_all),
        np.array(y_prob_all),
        pd.DataFrame(fold_results),
        selected_features_all
    )

In [ ]:

y_true, y_pred, y_prob, fold_df, selected_features_all = train_loso_cnn(
    X,
    y,
    groups,
    k_features=12,
    threshold=0.45,
    pd_weight=1.0
)

acc = accuracy_score(y_true, y_pred)
sens = recall_score(y_true, y_pred, pos_label=1)
spec = recall_score(y_true, y_pred, pos_label=0)
mcc = matthews_corrcoef(y_true, y_pred)

print("\n" + "="*60)
print("GENEL LOSO SONUÇLARI - İYİLEŞTİRİLMİŞ CNN")
print("="*60)
print(f"Accuracy   : {acc:.4f}")
print(f"Sensitivity: {sens:.4f}")
print(f"Specificity: {spec:.4f}")
print(f"MCC        : {mcc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))


FOLD 1/40
Test subject: 1
Gerçek: 1 Tahmin: 1 Olasılık: 0.7408677
Fold accuracy: 1.0

FOLD 2/40
Test subject: 2
Gerçek: 1 Tahmin: 1 Olasılık: 0.93381613
Fold accuracy: 1.0

FOLD 3/40
Test subject: 3
Gerçek: 1 Tahmin: 0 Olasılık: 0.31119254
Fold accuracy: 0.0

FOLD 4/40
Test subject: 4
Gerçek: 1 Tahmin: 1 Olasılık: 0.50170535
Fold accuracy: 1.0

FOLD 5/40
Test subject: 5
Gerçek: 1 Tahmin: 1 Olasılık: 0.7836464
Fold accuracy: 1.0

FOLD 6/40
Test subject: 6
Gerçek: 1 Tahmin: 1 Olasılık: 0.8290328
Fold accuracy: 1.0

FOLD 7/40
Test subject: 7
Gerçek: 1 Tahmin: 1 Olasılık: 0.87737155
Fold accuracy: 1.0

FOLD 8/40
Test subject: 8
Gerçek: 1 Tahmin: 1 Olasılık: 0.75812346
Fold accuracy: 1.0

FOLD 9/40
Test subject: 9
Gerçek: 1 Tahmin: 1 Olasılık: 0.71907747
Fold accuracy: 1.0

FOLD 10/40
Test subject: 10
Gerçek: 1 Tahmin: 0 Olasılık: 0.42987332
Fold accuracy: 0.0

FOLD 11/40
Test subject: 11
Gerçek: 1 Tahmin: 0 Olasılık: 0.43850657
Fold accuracy: 0.0

FOLD 12/40
Test subject: 12
Gerçek: 1 Tah

In [ ]:
y_true, y_pred, y_prob, fold_df, selected_features_all = train_loso_cnn(
    X,
    y,
    groups,
    k_features=15
)

acc = accuracy_score(y_true, y_pred)
sens = recall_score(y_true, y_pred, pos_label=1)
spec = recall_score(y_true, y_pred, pos_label=0)
mcc = matthews_corrcoef(y_true, y_pred)

print("\n" + "="*60)
print("GENEL SONUÇLAR")
print("="*60)
print(f"Accuracy   : {acc:.4f}")
print(f"Sensitivity: {sens:.4f}")
print(f"Specificity: {spec:.4f}")
print(f"MCC        : {mcc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))


FOLD 1/40
Test subject: 1
Gerçek: 1 Tahmin: 1 Olasılık: 0.9568506
Fold accuracy: 1.0

FOLD 2/40
Test subject: 2
Gerçek: 1 Tahmin: 1 Olasılık: 0.9915381
Fold accuracy: 1.0

FOLD 3/40
Test subject: 3
Gerçek: 1 Tahmin: 0 Olasılık: 0.1799
Fold accuracy: 0.0

FOLD 4/40
Test subject: 4
Gerçek: 1 Tahmin: 0 Olasılık: 0.32787007
Fold accuracy: 0.0

FOLD 5/40


Test subject: 5
Gerçek: 1 Tahmin: 1 Olasılık: 0.92318076
Fold accuracy: 1.0

FOLD 6/40


Test subject: 6
Gerçek: 1 Tahmin: 1 Olasılık: 0.5573905
Fold accuracy: 1.0

FOLD 7/40
Test subject: 7
Gerçek: 1 Tahmin: 1 Olasılık: 0.6462629
Fold accuracy: 1.0

FOLD 8/40
Test subject: 8
Gerçek: 1 Tahmin: 1 Olasılık: 0.88300925
Fold accuracy: 1.0

FOLD 9/40
Test subject: 9
Gerçek: 1 Tahmin: 1 Olasılık: 0.63197535
Fold accuracy: 1.0

FOLD 10/40
Test subject: 10
Gerçek: 1 Tahmin: 1 Olasılık: 0.56332475
Fold accuracy: 1.0

FOLD 11/40
Test subject: 11
Gerçek: 1 Tahmin: 0 Olasılık: 0.2582807
Fold accuracy: 0.0

FOLD 12/40
Test subject: 12
Gerçek: 1 Tahmin: 1 Olasılık: 0.9836886
Fold accuracy: 1.0

FOLD 13/40
Test subject: 13
Gerçek: 1 Tahmin: 1 Olasılık: 0.5817229
Fold accuracy: 1.0

FOLD 14/40
Test subject: 14
Gerçek: 1 Tahmin: 1 Olasılık: 0.6996954
Fold accuracy: 1.0

FOLD 15/40
Test subject: 15
Gerçek: 1 Tahmin: 0 Olasılık: 0.4988254
Fold accuracy: 0.0

FOLD 16/40
Test subject: 16
Gerçek: 1 Tahmin: 1 Olasılık: 0.74091583
Fold accuracy: 1.0

FOLD 17/40
Test subject: 17
Gerçek: 1 Tahmin: 

In [ ]:
# ============================================================
# KISA VE MANTIKLI ARAMA
# Önceki sonuçlara göre sadece 6 deneme
# ============================================================

quick_grid = [
    {"k": 10, "threshold": 0.45, "pd_weight": 1.0},
    {"k": 12, "threshold": 0.45, "pd_weight": 1.0},
    {"k": 15, "threshold": 0.45, "pd_weight": 1.0},

    {"k": 10, "threshold": 0.50, "pd_weight": 1.0},
    {"k": 12, "threshold": 0.50, "pd_weight": 1.0},
    {"k": 15, "threshold": 0.50, "pd_weight": 1.0},
]

quick_results = []

for i, p in enumerate(quick_grid, 1):
    print("\n" + "#"*60)
    print(f"DENEME {i}/{len(quick_grid)}")
    print(p)
    print("#"*60)

    y_true_q, y_pred_q, y_prob_q, fold_df_q, selected_features_q = train_loso_cnn(
        X,
        y,
        groups,
        k_features=p["k"],
        threshold=p["threshold"],
        pd_weight=p["pd_weight"]
    )

    acc_q = accuracy_score(y_true_q, y_pred_q)
    sens_q = recall_score(y_true_q, y_pred_q, pos_label=1)
    spec_q = recall_score(y_true_q, y_pred_q, pos_label=0)
    mcc_q = matthews_corrcoef(y_true_q, y_pred_q)

    quick_results.append({
        "k_features": p["k"],
        "threshold": p["threshold"],
        "pd_weight": p["pd_weight"],
        "accuracy": acc_q,
        "sensitivity": sens_q,
        "specificity": spec_q,
        "mcc": mcc_q
    })

quick_df = pd.DataFrame(quick_results)

quick_df.sort_values(
    by=["mcc", "accuracy"],
    ascending=False
)


############################################################
DENEME 1/6
{'k': 10, 'threshold': 0.45, 'pd_weight': 1.0}
############################################################

FOLD 1/40
Test subject: 1
Gerçek: 1 Tahmin: 1 Olasılık: 0.8598372
Fold accuracy: 1.0

FOLD 2/40
Test subject: 2
Gerçek: 1 Tahmin: 1 Olasılık: 0.91738814
Fold accuracy: 1.0

FOLD 3/40
Test subject: 3
Gerçek: 1 Tahmin: 0 Olasılık: 0.33475333
Fold accuracy: 0.0

FOLD 4/40
Test subject: 4
Gerçek: 1 Tahmin: 1 Olasılık: 0.70905685
Fold accuracy: 1.0

FOLD 5/40
Test subject: 5
Gerçek: 1 Tahmin: 1 Olasılık: 0.73550284
Fold accuracy: 1.0

FOLD 6/40
Test subject: 6
Gerçek: 1 Tahmin: 1 Olasılık: 0.6569761
Fold accuracy: 1.0

FOLD 7/40
Test subject: 7
Gerçek: 1 Tahmin: 1 Olasılık: 0.8908587
Fold accuracy: 1.0

FOLD 8/40
Test subject: 8
Gerçek: 1 Tahmin: 1 Olasılık: 0.88157207
Fold accuracy: 1.0

FOLD 9/40
Test subject: 9
Gerçek: 1 Tahmin: 1 Olasılık: 0.4507141
Fold accuracy: 1.0

FOLD 10/40
Test subject: 10
Gerçek: 1 T

,k_features,threshold,pd_weight,accuracy,sensitivity,specificity,mcc
1,12,0.45,1.0,0.850,0.85,0.85,0.700000
0,10,0.45,1.0,0.800,0.80,0.80,0.600000
3,10,0.50,1.0,0.750,0.65,0.85,0.510310
4,12,0.50,1.0,0.750,0.75,0.75,0.500000
2,15,0.45,1.0,0.675,0.60,0.75,0.354005
5,15,0.50,1.0,0.675,0.65,0.70,0.350438


In [ ]:
quick_df.sort_values(by=["mcc", "accuracy"], ascending=False)

,k_features,threshold,pd_weight,accuracy,sensitivity,specificity,mcc
1,12,0.45,1.0,0.850,0.85,0.85,0.700000
0,10,0.45,1.0,0.800,0.80,0.80,0.600000
3,10,0.50,1.0,0.750,0.65,0.85,0.510310
4,12,0.50,1.0,0.750,0.75,0.75,0.500000
2,15,0.45,1.0,0.675,0.60,0.75,0.354005
5,15,0.50,1.0,0.675,0.65,0.70,0.350438


In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_true, y_prob)

best_idx = np.argmax(tpr - fpr)
best_threshold = thresholds[best_idx]

print("En iyi threshold:", best_threshold)

En iyi threshold: 0.59638554


In [ ]:
new_thr = 0.59638554  # ~0.60 da deneyebilirsin
y_pred_new = (y_prob >= new_thr).astype(int)

from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report, matthews_corrcoef

acc = accuracy_score(y_true, y_pred_new)
sens = recall_score(y_true, y_pred_new, pos_label=1)
spec = recall_score(y_true, y_pred_new, pos_label=0)
mcc = matthews_corrcoef(y_true, y_pred_new)

print(f"Accuracy   : {acc:.4f}")
print(f"Sensitivity: {sens:.4f}")
print(f"Specificity: {spec:.4f}")
print(f"MCC        : {mcc:.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred_new))
print("\nClassification Report:\n", classification_report(y_true, y_pred_new))

Accuracy   : 0.7500
Sensitivity: 0.6500
Specificity: 0.8500
MCC        : 0.5103

Confusion Matrix:
 [[17  3]
 [ 7 13]]

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.85      0.77        20
           1       0.81      0.65      0.72        20

    accuracy                           0.75        40
   macro avg       0.76      0.75      0.75        40
weighted avg       0.76      0.75      0.75        40



In [ ]:
# ============================================================
# 1. EN SIK SEÇİLEN 12 FEATURE'I SABİTLE
# ============================================================

from collections import Counter

feature_counts = Counter(selected_features_all)

feature_df = pd.DataFrame({
    "feature": list(feature_counts.keys()),
    "selected_count": list(feature_counts.values())
}).sort_values("selected_count", ascending=False)

feature_df["selection_rate"] = feature_df["selected_count"] / len(np.unique(groups))

top_features = feature_df.head(12)["feature"].tolist()

print("Seçilen sabit feature'lar:")
for f in top_features:
    print(f)

X_fixed = summary_df[top_features].values.astype(np.float32)
y_fixed = summary_df["class"].values.astype(int)
groups_fixed = summary_df["subject_id"].values

print("Yeni X boyutu:", X_fixed.shape)

Seçilen sabit feature'lar:
Shimmer_apq11_wmean
Frac_unvoiced_frames_wstd
Degree_voice_breaks_wstd
Degree_voice_breaks_wmean
Num_voice_breaks_wstd
Num_voice_breaks_wmean
HTN_wstd
Mean_pitch_wstd
NTH_wstd
Frac_unvoiced_frames_wmean
Mean_period_wstd
Num_periods_wmean
Yeni X boyutu: (40, 12)


In [ ]:
import joblib, os

os.makedirs("model", exist_ok=True)

y_true_final, y_pred_final, y_prob_final, fold_df_final, _ = train_loso_cnn(
    X, y, groups,
    k_features=12,
    threshold=0.45,
    pd_weight=1.0
)

joblib.dump(fold_df_final, "model/conv1d_fold_results.pkl")
joblib.dump(summary_df,    "model/summary_df_conv1d.pkl")

print("Kaydedildi.")


FOLD 1/40
Test subject: 1
Gerçek: 1 Tahmin: 1 Olasılık: 0.49973726
Fold accuracy: 1.0

FOLD 2/40
Test subject: 2
Gerçek: 1 Tahmin: 1 Olasılık: 0.80591965
Fold accuracy: 1.0

FOLD 3/40
Test subject: 3
Gerçek: 1 Tahmin: 0 Olasılık: 0.39954242
Fold accuracy: 0.0

FOLD 4/40
Test subject: 4
Gerçek: 1 Tahmin: 0 Olasılık: 0.4133709
Fold accuracy: 0.0

FOLD 5/40
Test subject: 5
Gerçek: 1 Tahmin: 1 Olasılık: 0.8458162
Fold accuracy: 1.0

FOLD 6/40
Test subject: 6
Gerçek: 1 Tahmin: 1 Olasılık: 0.48982364
Fold accuracy: 1.0

FOLD 7/40
Test subject: 7
Gerçek: 1 Tahmin: 1 Olasılık: 0.76814824
Fold accuracy: 1.0

FOLD 8/40
Test subject: 8
Gerçek: 1 Tahmin: 1 Olasılık: 0.9723196
Fold accuracy: 1.0

FOLD 9/40
Test subject: 9
Gerçek: 1 Tahmin: 1 Olasılık: 0.5671878
Fold accuracy: 1.0

FOLD 10/40
Test subject: 10
Gerçek: 1 Tahmin: 0 Olasılık: 0.19660139
Fold accuracy: 0.0

FOLD 11/40
Test subject: 11
Gerçek: 1 Tahmin: 1 Olasılık: 0.6316006
Fold accuracy: 1.0

FOLD 12/40
Test subject: 12
Gerçek: 1 Tahmi

In [ ]:
import joblib
import os

os.makedirs("model", exist_ok=True)

joblib.dump(fold_df, "model/conv1d_fold_results.pkl")
joblib.dump(summary_df, "model/summary_df_conv1d.pkl")

print("✓ Kaydedildi.")
print(fold_df)

from google.colab import files
files.download("model/conv1d_fold_results.pkl")
files.download("model/summary_df_conv1d.pkl")

✓ Kaydedildi.
    fold  subject_id  true  pred      prob  acc
0      1           1     1     1  0.740868  1.0
1      2           2     1     1  0.933816  1.0
2      3           3     1     0  0.311193  0.0
3      4           4     1     1  0.501705  1.0
4      5           5     1     1  0.783646  1.0
5      6           6     1     1  0.829033  1.0
6      7           7     1     1  0.877372  1.0
7      8           8     1     1  0.758123  1.0
8      9           9     1     1  0.719077  1.0
9     10          10     1     0  0.429873  0.0
10    11          11     1     0  0.438507  0.0
11    12          12     1     1  0.508045  1.0
12    13          13     1     1  0.802429  1.0
13    14          14     1     0  0.435867  0.0
14    15          15     1     1  0.660923  1.0
15    16          16     1     1  0.806690  1.0
16    17          17     1     1  0.807457  1.0
17    18          18     1     0  0.121246  0.0
18    19          19     1     1  0.722561  1.0
19    20          20     1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>